# AI_Agent_Workshop_Day2_02 — Build the Service Agent

In this notebook we compare three increasingly capable systems:

1. a prompt-only baseline,
2. a retrieval-augmented baseline,
3. a tool-using Gemini agent.

The goal is to help students see why agentified design is useful.

## Setup

This notebook uses a local curated CSV/JSON retrieval path by default.

To run Gemini cells, set your API key in the environment before launching Jupyter.

In [36]:
from pathlib import Path
import json
import os
import re
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("day2_workshop")

DATA_DIR = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"
catalog = pd.read_csv(DATA_DIR / "service_catalog.csv")
catalog.head()

,service_name,jurisdiction_level,responsible_body,description,keywords,next_steps_hint,source_url
0,garbage pickup,Region,Region of Waterloo Waste Management,"Residential garbage collection schedule, misse...",garbage;waste;pickup;missed collection;curbside,Check the regional waste collection schedule; ...,https://www.regionofwaterloo.ca/
1,recycling collection,Region,Region of Waterloo Waste Management,Blue box recycling collection and schedule inf...,recycling;blue box;collection,Check the regional collection calendar and rec...,https://www.regionofwaterloo.ca/
2,property tax billing,City,City of Kitchener Revenue Division,"Property tax billing, payment options, and acc...",property tax;tax bill;billing;assessment,Visit the city's property tax page or contact ...,https://www.kitchener.ca/
3,water billing,City,City of Kitchener Utilities,Water billing and account inquiries for city u...,water bill;utilities;billing,Check your city utility account and payment op...,https://www.kitchener.ca/
4,snow removal on regional roads,Region,Region of Waterloo Transportation Services,Maintenance and snow clearing for regional roads.,snow removal;plowing;regional roads;winter mai...,Confirm whether the road is regional before co...,https://www.regionofwaterloo.ca/


In [37]:
GEMINI_AVAILABLE = False

try:
    from google import genai
    from google.genai import types
    api_key = os.getenv("GEMINI_API_KEY")
    if api_key:
        client = genai.Client(api_key=api_key)
        GEMINI_AVAILABLE = True
except Exception as exc:
    print("Gemini SDK not available:", exc)

print("Gemini available:", GEMINI_AVAILABLE)

Gemini available: True


## Step 1 — Prompt-only baseline

In [38]:
def baseline_prompt(question: str) -> str:
    return f'''
    You are a public-service routing assistant for residents in Kitchener and Waterloo Region.

    Return a JSON object with the keys:
    service_name, jurisdiction_level, responsible_body, confidence, reasoning_summary, next_steps, sources

    Question: {question}
    '''.strip()

sample_question = "Who do I contact about garbage pickup?"
print(baseline_prompt(sample_question))

You are a public-service routing assistant for residents in Kitchener and Waterloo Region.

    Return a JSON object with the keys:
    service_name, jurisdiction_level, responsible_body, confidence, reasoning_summary, next_steps, sources

    Question: Who do I contact about garbage pickup?


In [39]:
if GEMINI_AVAILABLE:
    baseline_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=baseline_prompt(sample_question),
    )
    print(baseline_response.text)
else:
    print("Skipping live model call. Discuss likely failure modes instead.")

```json
{
  "service_name": "Garbage Pickup (Waste Collection)",
  "jurisdiction_level": "Regional",
  "responsible_body": "Region of Waterloo",
  "confidence": "High",
  "reasoning_summary": "In Kitchener and the broader Waterloo Region, waste management services, including garbage pickup, are managed and provided by the Regional Municipality of Waterloo, not the individual cities (like Kitchener or Waterloo). This is a common division of services in a two-tier municipal system.",
  "next_steps": "To get information about garbage pickup schedules, missed collections, waste sorting guidelines, or to report an issue:\n\n1.  **Visit the Region of Waterloo Waste Management Website:** This is the most comprehensive resource. Search for 'Region of Waterloo waste' or 'garbage collection'.\n2.  **Use the 'Waste Whiz' Tool:** The Region offers an online tool and mobile app called 'Waste Whiz' where you can type in your address to get your specific collection schedule, sign up for reminders, an

## Step 2 — Local retrieval over the curated catalog

In [40]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def keyword_retrieve(query: str, df: pd.DataFrame, top_k: int = 3) -> pd.DataFrame:
    q_tokens = set(tokenize(query))
    scored_rows = []
    for _, row in df.iterrows():
        text = " ".join([
            str(row.get("service_name", "")),
            str(row.get("description", "")),
            str(row.get("keywords", "")),
            str(row.get("responsible_body", "")),
        ]).lower()
        row_tokens = tokenize(text)
        overlap = sum(token in row_tokens for token in q_tokens)
        scored_rows.append((overlap, row.to_dict()))
    ranked = sorted(scored_rows, key=lambda x: x[0], reverse=True)
    ranked = [row for score, row in ranked if score > 0][:top_k]
    return pd.DataFrame(ranked)

retrieved = keyword_retrieve(sample_question, catalog, top_k=3)
retrieved

,service_name,jurisdiction_level,responsible_body,description,keywords,next_steps_hint,source_url
0,garbage pickup,Region,Region of Waterloo Waste Management,"Residential garbage collection schedule, misse...",garbage;waste;pickup;missed collection;curbside,Check the regional waste collection schedule; ...,https://www.regionofwaterloo.ca/


In [41]:
def build_grounded_prompt(question: str, retrieved_df: pd.DataFrame) -> str:
    context = retrieved_df.to_dict(orient="records")
    return f'''
    You are a public-service routing assistant.

    Use the retrieved service catalog entries below as your primary evidence.
    If the evidence is ambiguous, say so.

    Retrieved context:
    {json.dumps(context, indent=2)}

    Return a JSON object with:
    service_name, jurisdiction_level, responsible_body, confidence, reasoning_summary, next_steps, sources

    Question: {question}
    '''.strip()

grounded_prompt = build_grounded_prompt(sample_question, retrieved)
print(grounded_prompt[:1200])

You are a public-service routing assistant.

    Use the retrieved service catalog entries below as your primary evidence.
    If the evidence is ambiguous, say so.

    Retrieved context:
    [
  {
    "service_name": "garbage pickup",
    "jurisdiction_level": "Region",
    "responsible_body": "Region of Waterloo Waste Management",
    "description": "Residential garbage collection schedule, missed pickup, and waste cart information.",
    "keywords": "garbage;waste;pickup;missed collection;curbside",
    "next_steps_hint": "Check the regional waste collection schedule; report a missed pickup if applicable",
    "source_url": "https://www.regionofwaterloo.ca/"
  }
]

    Return a JSON object with:
    service_name, jurisdiction_level, responsible_body, confidence, reasoning_summary, next_steps, sources

    Question: Who do I contact about garbage pickup?


In [46]:
# Safe version of the grounded call demo

if GEMINI_AVAILABLE:
    try:
        grounded_response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=grounded_prompt,
        )
        print(grounded_response.text)
    except Exception as e:
        print("Grounded call failed:", type(e).__name__)
        print(e)
else:
    print("Skipping live grounded call.")

```json
{
  "service_name": "garbage pickup",
  "jurisdiction_level": "Region",
  "responsible_body": "Region of Waterloo Waste Management",
  "confidence": "High",
  "reasoning_summary": "The retrieved service catalog entry directly matches 'garbage pickup' and clearly identifies the 'Region of Waterloo Waste Management' as the responsible body for this service.",
  "next_steps": "Check the regional waste collection schedule; report a missed pickup if applicable.",
  "sources": [
    "https://www.regionofwaterloo.ca/"
  ]
}
```


## Step 3 — Convert the application into a tool-using agent

In [47]:
def search_service_index(query: str) -> list[dict]:
    results = keyword_retrieve(query, catalog, top_k=3)
    return results.to_dict(orient="records")

def lookup_service_owner(service_name: str) -> dict:
    matches = catalog[catalog["service_name"].str.lower() == service_name.lower()]
    if len(matches) == 0:
        return {
            "service_name": service_name,
            "jurisdiction_level": "Unclear",
            "responsible_body": "Unknown",
            "reasoning_summary": "No exact service match found in the local catalog.",
            "sources": [],
        }
    row = matches.iloc[0]
    return {
        "service_name": row["service_name"],
        "jurisdiction_level": row["jurisdiction_level"],
        "responsible_body": row["responsible_body"],
        "reasoning_summary": row["description"],
        "sources": [row["source_url"]],
    }

def suggest_next_steps(jurisdiction_level: str, service_name: str) -> dict:
    matches = catalog[catalog["service_name"].str.lower() == service_name.lower()]
    if len(matches) == 0:
        return {"next_steps": ["Search the relevant official service page before contacting an office."]}
    row = matches.iloc[0]
    return {
        "next_steps": [
            row["next_steps_hint"],
            f"Verify current details on the official source: {row['source_url']}"
        ]
    }

print(search_service_index("garbage pickup"))
print(lookup_service_owner("garbage pickup"))
print(suggest_next_steps("Region", "garbage pickup"))

[{'service_name': 'garbage pickup', 'jurisdiction_level': 'Region', 'responsible_body': 'Region of Waterloo Waste Management', 'description': 'Residential garbage collection schedule, missed pickup, and waste cart information.', 'keywords': 'garbage;waste;pickup;missed collection;curbside', 'next_steps_hint': 'Check the regional waste collection schedule; report a missed pickup if applicable', 'source_url': 'https://www.regionofwaterloo.ca/'}]
{'service_name': 'garbage pickup', 'jurisdiction_level': 'Region', 'responsible_body': 'Region of Waterloo Waste Management', 'reasoning_summary': 'Residential garbage collection schedule, missed pickup, and waste cart information.', 'sources': ['https://www.regionofwaterloo.ca/']}
{'next_steps': ['Check the regional waste collection schedule; report a missed pickup if applicable', 'Verify current details on the official source: https://www.regionofwaterloo.ca/']}


## Gemini tool declarations

In [48]:
if GEMINI_AVAILABLE:
    tool_declarations = [
        types.Tool(function_declarations=[{
            "name": "search_service_index",
            "description": "Search the local service catalog for relevant public services.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Resident question or service search string"}
                },
                "required": ["query"]
            }
        }]),
        types.Tool(function_declarations=[{
            "name": "lookup_service_owner",
            "description": "Look up the responsible level of government and body for a service.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service_name": {"type": "string", "description": "Canonical service name"}
                },
                "required": ["service_name"]
            }
        }]),
        types.Tool(function_declarations=[{
            "name": "suggest_next_steps",
            "description": "Suggest next steps for a service after ownership is known.",
            "parameters": {
                "type": "object",
                "properties": {
                    "jurisdiction_level": {"type": "string"},
                    "service_name": {"type": "string"}
                },
                "required": ["jurisdiction_level", "service_name"]
            }
        }]),
    ]
    print(tool_declarations[0])
else:
    print("Tool declarations shown only when Gemini SDK is available.")

retrieval=None computer_use=None file_search=None google_search=None google_maps=None code_execution=None enterprise_web_search=None function_declarations=[FunctionDeclaration(
  description='Search the local service catalog for relevant public services.',
  name='search_service_index',
  parameters=Schema(
    properties={
      'query': Schema(
        description='Resident question or service search string',
        type=<Type.STRING: 'STRING'>
      )
    },
    required=[
      'query',
    ],
    type=<Type.OBJECT: 'OBJECT'>
  )
)] google_search_retrieval=None parallel_ai_search=None url_context=None mcp_servers=None


## Manual tool loop

In [49]:
TOOL_REGISTRY = {
    "search_service_index": search_service_index,
    "lookup_service_owner": lookup_service_owner,
    "suggest_next_steps": suggest_next_steps,
}

def execute_tool_call(function_call):
    fn_name = function_call.name
    fn_args = dict(function_call.args)
    print("Tool selected:", fn_name)
    print("Arguments:", fn_args)
    result = TOOL_REGISTRY[fn_name](**fn_args)
    print("Tool result:", result)
    return fn_name, result

In [50]:
question = "Who do I contact about garbage pickup?"

if GEMINI_AVAILABLE:
    initial_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f'''
        You are a public-service routing assistant for residents in Kitchener / Waterloo Region.

        Use tools when useful. Prefer grounded, structured answers.
        User question: {question}
        ''',
        config=types.GenerateContentConfig(
            tools=tool_declarations,
        ),
    )

    parts = initial_response.candidates[0].content.parts
    parts
else:
    print("Skipping live tool-call generation.")

In [51]:
if GEMINI_AVAILABLE:
    part = initial_response.candidates[0].content.parts[0]

    if getattr(part, "function_call", None):
        fn_name, tool_result = execute_tool_call(part.function_call)
    else:
        tool_result = None
        print("No function call returned.")
        print(initial_response.text)

Tool selected: search_service_index
Arguments: {'query': 'garbage pickup'}
Tool result: [{'service_name': 'garbage pickup', 'jurisdiction_level': 'Region', 'responsible_body': 'Region of Waterloo Waste Management', 'description': 'Residential garbage collection schedule, missed pickup, and waste cart information.', 'keywords': 'garbage;waste;pickup;missed collection;curbside', 'next_steps_hint': 'Check the regional waste collection schedule; report a missed pickup if applicable', 'source_url': 'https://www.regionofwaterloo.ca/'}]


## Return the tool result back to the model

In [53]:
# Return the tool result back to the model

if GEMINI_AVAILABLE and tool_result is not None:
    # The Gemini SDK expects response= to be a dictionary.
    # Some tools may return a list, so we wrap it in a dictionary.
    safe_tool_result = tool_result if isinstance(tool_result, dict) else {"results": tool_result}

    final_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            types.Content(
                role="user",
                parts=[types.Part(text=question)]
            ),
            types.Content(
                role="model",
                parts=[part]
            ),
            types.Content(
                role="user",
                parts=[
                    types.Part.from_function_response(
                        name=fn_name,
                        response=safe_tool_result
                    )
                ]
            )
        ],
    )

    print(final_response.text)
else:
    print("Skipping final synthesis call.")

You should contact the **Region of Waterloo Waste Management**.

They handle:
*   Residential garbage collection schedules
*   Missed pickups
*   Waste cart information

You can find more details on their website: https://www.regionofwaterloo.ca/


## Exercise

1. Add a new tool called `retrieve_service_docs`.
2. Make the final answer always include at least one official source URL.
3. Create a question that requires hedging or clarification.
4. Compare the prompt-only baseline against the tool-using version on five examples.

### Exercise 1: Add a New Tool Called `retrieve_service_docs`

In this step, we add a new tool called `retrieve_service_docs`.
This tool retrieves fuller service documentation from the local service catalog, including the description, next step hints, and official source URL.
It helps the agent provide more grounded and informative answers.

In [54]:
# Retrieve fuller service documentation from the local catalog
def retrieve_service_docs(service_name: str) -> dict:
    # Find matching services by exact service name
    matches = catalog[catalog["service_name"].str.lower() == service_name.lower()]
    
    # Return a fallback response if no exact match is found
    if len(matches) == 0:
        return {
            "service_name": service_name,
            "description": "No matching service documentation was found in the local catalog.",
            "responsible_body": "Unknown",
            "next_steps_hint": "Search the official municipal website for the latest information.",
            "source_url": ""
        }
    
    # Use the first matched row
    row = matches.iloc[0]
    
    # Return a structured documentation record
    return {
        "service_name": row["service_name"],
        "description": row["description"],
        "responsible_body": row["responsible_body"],
        "next_steps_hint": row["next_steps_hint"],
        "source_url": row["source_url"]
    }

# Test the new tool
print(retrieve_service_docs("garbage pickup"))

{'service_name': 'garbage pickup', 'description': 'Residential garbage collection schedule, missed pickup, and waste cart information.', 'responsible_body': 'Region of Waterloo Waste Management', 'next_steps_hint': 'Check the regional waste collection schedule; report a missed pickup if applicable', 'source_url': 'https://www.regionofwaterloo.ca/'}


## Exercise 2: Always Include at Least One Official Source URL

In this step, we update the tool registry and the answer generation logic so that the final response always includes at least one official source URL whenever it is available.

This makes the agent more grounded and trustworthy because the answer is supported by an official public source.

In [57]:
# Exercise 2: update the registry and final answer logic
# Goal: always include at least one official source URL in the final answer

# Update the tool registry to include the new tool created in Exercise 1
TOOL_REGISTRY = {
    "search_service_index": search_service_index,
    "lookup_service_owner": lookup_service_owner,
    "suggest_next_steps": suggest_next_steps,
    "retrieve_service_docs": retrieve_service_docs,
}

# If Gemini is available, update the tool declarations as well
if GEMINI_AVAILABLE:
    tool_declarations = [
        types.Tool(function_declarations=[{
            "name": "search_service_index",
            "description": "Search the local service catalog for relevant public services.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Resident question or service search string"
                    }
                },
                "required": ["query"]
            }
        }]),
        types.Tool(function_declarations=[{
            "name": "lookup_service_owner",
            "description": "Look up the responsible level of government and body for a service.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service_name": {
                        "type": "string",
                        "description": "Canonical service name"
                    }
                },
                "required": ["service_name"]
            }
        }]),
        types.Tool(function_declarations=[{
            "name": "suggest_next_steps",
            "description": "Suggest next steps for a service after ownership is known.",
            "parameters": {
                "type": "object",
                "properties": {
                    "jurisdiction_level": {"type": "string"},
                    "service_name": {"type": "string"}
                },
                "required": ["jurisdiction_level", "service_name"]
            }
        }]),
        types.Tool(function_declarations=[{
            "name": "retrieve_service_docs",
            "description": "Retrieve fuller service documentation from the local catalog, including description, responsible body, next steps, and official source URL.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service_name": {
                        "type": "string",
                        "description": "Canonical service name"
                    }
                },
                "required": ["service_name"]
            }
        }]),
    ]

    print("Updated tools loaded:", [fd.function_declarations[0].name for fd in tool_declarations])
else:
    print("Gemini not available, but TOOL_REGISTRY now includes retrieve_service_docs.")


# Helper function:
# Extract official source URLs from the tool result
def extract_official_urls(tool_result) -> list[str]:
    urls = []

    def walk(obj):
        if isinstance(obj, dict):
            for key, value in obj.items():
                if key in {"source_url", "sources"}:
                    if isinstance(value, str) and value.strip():
                        urls.append(value.strip())
                    elif isinstance(value, list):
                        for item in value:
                            if isinstance(item, str) and item.strip():
                                urls.append(item.strip())
                walk(value)
        elif isinstance(obj, list):
            for item in obj:
                walk(item)

    walk(tool_result)

    # Remove duplicates while keeping original order
    deduped = []
    for url in urls:
        if url not in deduped:
            deduped.append(url)

    return deduped


# Main tool-using agent function
def run_tool_agent(question: str):
    # Return a fallback message if Gemini SDK or API key is missing
    if not GEMINI_AVAILABLE:
        return {
            "question": question,
            "status": "Gemini SDK or API key not available"
        }

    # First response: let Gemini decide whether to call a tool
    initial_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"""
        You are a public-service routing assistant for residents in Kitchener / Waterloo Region.

        Use tools when useful. Prefer grounded, structured answers.
        If the answer is uncertain or jurisdiction depends on missing details, say so clearly.
        User question: {question}
        """.strip(),
        config=types.GenerateContentConfig(
            tools=tool_declarations,
        ),
    )

    parts = initial_response.candidates[0].content.parts
    first_part = parts[0]

    # If no tool was called, return the direct model answer
    if not getattr(first_part, "function_call", None):
        return {
            "question": question,
            "final_text": initial_response.text,
            "tool_name": None,
            "tool_result": None,
            "official_urls": []
        }

    # Execute the tool call
    fn_name, tool_result = execute_tool_call(first_part.function_call)

    # Extract official URLs from the tool result
    official_urls = extract_official_urls(tool_result)

    # IMPORTANT:
    # Gemini expects the function response to be a dictionary.
    # Some tools return a list, so we wrap list outputs in a dictionary.
    if isinstance(tool_result, dict):
        safe_tool_result = tool_result
    elif isinstance(tool_result, list):
        safe_tool_result = {"results": tool_result}
    else:
        safe_tool_result = {"result": str(tool_result)}

    # Second response: generate the final resident-facing answer
    final_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            types.Content(
                role="user",
                parts=[types.Part(text=question)]
            ),
            types.Content(
                role="model",
                parts=[first_part]
            ),
            types.Content(
                role="user",
                parts=[
                    types.Part.from_function_response(
                        name=fn_name,
                        response=safe_tool_result
                    ),
                    types.Part(text=f"""
                    Write a concise final answer for the resident.

                    Requirements:
                    - Always include at least one official source URL from the tool result when available.
                    - If the evidence is incomplete or jurisdiction depends on missing details, hedge appropriately.
                    - Mention the responsible body if known.
                    - Suggest a practical next step.

                    Available official URLs: {official_urls}
                    """.strip())
                ]
            )
        ],
    )

    # Return the structured result
    return {
        "question": question,
        "tool_name": fn_name,
        "tool_result": tool_result,
        "official_urls": official_urls,
        "final_text": final_response.text
    }


# Quick test for Exercise 2
exercise2_demo = run_tool_agent("Who do I contact about garbage pickup?")
exercise2_demo

Updated tools loaded: ['search_service_index', 'lookup_service_owner', 'suggest_next_steps', 'retrieve_service_docs']
Tool selected: search_service_index
Arguments: {'query': 'garbage pickup'}
Tool result: [{'service_name': 'garbage pickup', 'jurisdiction_level': 'Region', 'responsible_body': 'Region of Waterloo Waste Management', 'description': 'Residential garbage collection schedule, missed pickup, and waste cart information.', 'keywords': 'garbage;waste;pickup;missed collection;curbside', 'next_steps_hint': 'Check the regional waste collection schedule; report a missed pickup if applicable', 'source_url': 'https://www.regionofwaterloo.ca/'}]


{'question': 'Who do I contact about garbage pickup?',
 'tool_name': 'search_service_index',
 'tool_result': [{'service_name': 'garbage pickup',
   'jurisdiction_level': 'Region',
   'responsible_body': 'Region of Waterloo Waste Management',
   'description': 'Residential garbage collection schedule, missed pickup, and waste cart information.',
   'keywords': 'garbage;waste;pickup;missed collection;curbside',
   'next_steps_hint': 'Check the regional waste collection schedule; report a missed pickup if applicable',
   'source_url': 'https://www.regionofwaterloo.ca/'}],
 'official_urls': ['https://www.regionofwaterloo.ca/'],
 'final_text': 'You should contact the **Region of Waterloo Waste Management** about garbage pickup.\n\nFor more information, including the collection schedule, or to report a missed pickup, visit their website: https://www.regionofwaterloo.ca/'}

## Exercise 3: Create a Question That Requires Hedging or Clarification

In this step, we create a question that cannot be answered with full certainty from the service catalog alone.

A good example is a question where the responsible authority depends on missing details.  
This encourages the agent to hedge, clarify uncertainty, and avoid overconfident answers.

In [58]:
# Exercise 3: create a question that requires hedging or clarification

# This question is intentionally ambiguous.
# "My street" may refer to a regional road or a local city street.
# The agent should hedge instead of giving an overconfident answer.
hedging_question = "Who should I contact about snow removal on my street?"

print("Exercise 3 Question:")
print(hedging_question)

Exercise 3 Question:
Who should I contact about snow removal on my street?


In [59]:
# Run the tool agent using the ambiguous question

exercise3_demo = run_tool_agent(hedging_question)

# Show the full structured output
exercise3_demo

Tool selected: search_service_index
Arguments: {'query': 'snow removal'}
Tool result: [{'service_name': 'snow removal on regional roads', 'jurisdiction_level': 'Region', 'responsible_body': 'Region of Waterloo Transportation Services', 'description': 'Maintenance and snow clearing for regional roads.', 'keywords': 'snow removal;plowing;regional roads;winter maintenance', 'next_steps_hint': 'Confirm whether the road is regional before contacting the Region', 'source_url': 'https://www.regionofwaterloo.ca/'}, {'service_name': 'snow removal on city streets', 'jurisdiction_level': 'City', 'responsible_body': 'City of Kitchener Operations', 'description': 'Maintenance and snow clearing for local city streets and sidewalks where applicable.', 'keywords': 'snow removal;city streets;plowing;roads', 'next_steps_hint': 'Check the city operations page for winter maintenance details', 'source_url': 'https://www.kitchener.ca/'}]


{'question': 'Who should I contact about snow removal on my street?',
 'tool_name': 'search_service_index',
 'tool_result': [{'service_name': 'snow removal on regional roads',
   'jurisdiction_level': 'Region',
   'responsible_body': 'Region of Waterloo Transportation Services',
   'description': 'Maintenance and snow clearing for regional roads.',
   'keywords': 'snow removal;plowing;regional roads;winter maintenance',
   'next_steps_hint': 'Confirm whether the road is regional before contacting the Region',
   'source_url': 'https://www.regionofwaterloo.ca/'},
  {'service_name': 'snow removal on city streets',
   'jurisdiction_level': 'City',
   'responsible_body': 'City of Kitchener Operations',
   'description': 'Maintenance and snow clearing for local city streets and sidewalks where applicable.',
   'keywords': 'snow removal;city streets;plowing;roads',
   'next_steps_hint': 'Check the city operations page for winter maintenance details',
   'source_url': 'https://www.kitchener.c

In [60]:
# Print final answer safely

if "final_text" in exercise3_demo:
    print(exercise3_demo["final_text"])
else:
    print("No final_text found.")
    print(exercise3_demo)


The user needs to know who to contact about snow removal on their street.

The tool results indicate that there are two main possibilities, depending on whether the street is a regional road or a local city street:

1.  **Regional Roads:** Contact the **Region of Waterloo Transportation Services**.
    *   Next step: "Confirm whether the road is regional before contacting the Region".
    *   Source URL: `https://www.regionofwaterloo.ca/`

2.  **City Streets:** Contact the **City of Kitchener Operations**.
    *   Next step: "Check the city operations page for winter maintenance details".
    *   Source URL: `https://www.kitchener.ca/`

Therefore, the answer needs to:
*   Explain the two possibilities.
*   Advise the user to determine if their street is regional or local.
*   Provide the relevant contact body and URL for each.
*   Suggest the practical next steps.It depends on whether your street is a regional road or a local city street.

*   **For regional roads:** You should contac

## Exercise 4: Compare the Prompt-Only Baseline Against the Tool-Using Version

In this step, we compare a prompt-only baseline with the tool-using agent on five example questions.

The goal is to observe how tool use improves grounding, source attribution, and answer quality.  
The tool-using version should provide more reliable answers, include official source URLs, and handle ambiguity more carefully when needed.

In [62]:
# Exercise 4: prompt-only baseline function
# This function asks Gemini to answer directly without using any tools.

def run_prompt_only_baseline(question: str):
    if not GEMINI_AVAILABLE:
        return {
            "question": question,
            "status": "Gemini SDK or API key not available"
        }

    baseline_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"""
        You are a public-service routing assistant for residents in Kitchener / Waterloo Region.

        Answer the resident's question directly using only the model's own reasoning.
        Do not use tools.
        If you are uncertain, say so clearly.

        User question: {question}
        """.strip()
    )

    return {
        "question": question,
        "final_text": baseline_response.text
    }

In [63]:
# Five example questions for comparison

comparison_questions = [
    "Who do I contact about garbage pickup?",
    "Who handles water billing?",
    "Who should I contact about property tax billing?",
    "Who should I contact about snow removal on my street?",
    "Who handles recycling collection?"
]

print("Questions for Exercise 4:")
for i, q in enumerate(comparison_questions, start=1):
    print(f"{i}. {q}")

Questions for Exercise 4:
1. Who do I contact about garbage pickup?
2. Who handles water billing?
3. Who should I contact about property tax billing?
4. Who should I contact about snow removal on my street?
5. Who handles recycling collection?


In [66]:
# Run both the prompt-only baseline and the tool-using version
# Use try/except so one temporary API failure does not stop the whole comparison.

comparison_results = []

for question in comparison_questions:
    print("=" * 100)
    print("Question:", question)

    # Prompt-only baseline
    try:
        baseline_result = run_prompt_only_baseline(question)
    except Exception as e:
        baseline_result = {
            "question": question,
            "status": f"Baseline failed: {type(e).__name__}: {e}"
        }

    # Tool-using version
    try:
        tool_result = run_tool_agent(question)
    except Exception as e:
        tool_result = {
            "question": question,
            "status": f"Tool agent failed: {type(e).__name__}: {e}",
            "tool_name": None,
            "official_urls": []
        }

    comparison_results.append({
        "question": question,
        "prompt_only_answer": baseline_result.get("final_text", baseline_result.get("status", "No output")),
        "tool_used": tool_result.get("tool_name", None),
        "tool_agent_answer": tool_result.get("final_text", tool_result.get("status", "No output")),
        "official_urls": ", ".join(tool_result.get("official_urls", [])) if tool_result.get("official_urls") else ""
    })

print("Comparison run completed.")

Question: Who do I contact about garbage pickup?
Question: Who handles water billing?
Question: Who should I contact about property tax billing?
Question: Who should I contact about snow removal on my street?
Question: Who handles recycling collection?
Comparison run completed.


In [67]:
# Convert comparison results into a DataFrame for easier viewing

comparison_df = pd.DataFrame(comparison_results)

comparison_df

,question,prompt_only_answer,tool_used,tool_agent_answer,official_urls
0,Who do I contact about garbage pickup?,Baseline failed: ClientError: 429 RESOURCE_EXH...,None,Tool agent failed: ClientError: 429 RESOURCE_E...,
1,Who handles water billing?,Baseline failed: ClientError: 429 RESOURCE_EXH...,None,Tool agent failed: ClientError: 429 RESOURCE_E...,
2,Who should I contact about property tax billing?,Baseline failed: ClientError: 429 RESOURCE_EXH...,None,Tool agent failed: ClientError: 429 RESOURCE_E...,
3,Who should I contact about snow removal on my ...,Baseline failed: ClientError: 429 RESOURCE_EXH...,None,Tool agent failed: ClientError: 429 RESOURCE_E...,
4,Who handles recycling collection?,Baseline failed: ClientError: 429 RESOURCE_EXH...,None,Tool agent failed: ClientError: 429 RESOURCE_E...,


In [68]:
# Display a shorter comparison table

comparison_df[[
    "question",
    "tool_used",
    "official_urls"
]]

,question,tool_used,official_urls
0,Who do I contact about garbage pickup?,None,
1,Who handles water billing?,None,
2,Who should I contact about property tax billing?,None,
3,Who should I contact about snow removal on my ...,None,
4,Who handles recycling collection?,None,


In [69]:
# Print detailed side-by-side comparison for each example

for i, row in comparison_df.iterrows():
    print("\n" + "=" * 120)
    print(f"Example {i+1}")
    print(f"Question: {row['question']}\n")

    print("Prompt-only baseline answer:")
    print(row["prompt_only_answer"])
    print("\n" + "-" * 120)

    print("Tool-using answer:")
    print(row["tool_agent_answer"])
    print("\n" + "-" * 120)

    print("Tool used:")
    print(row["tool_used"])

    print("\nOfficial source URLs:")
    print(row["official_urls"] if row["official_urls"] else "No official URL returned")


Example 1
Question: Who do I contact about garbage pickup?

Prompt-only baseline answer:
Baseline failed: ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 38.057006888s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequests